In [14]:
# Author: Ethan Stroberg
# National Severe Storms Laboratory
# This code last updated on Mar 3, 2025
# This version only uses one user's classifications
#
# Version B1.6 - narrowed down to only use inputs from one user
# Version 1.5 - added T, RH, and P data from .csv and .txt files, includes addition of functions
# Version 1.4 - updated to give a user-based description of classifications
# Version 1.3 - updated to only use majority rules cases, normalized dataset
# Verison 1.2 - added machine learning models
# Version 1.1 - updated to only take in values classified as 2 or less
# Version 1.0 - original code written by Mike to read in data and plot histograms
#


In [1]:
#imports
import pandas as pd
import h5py
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc, precision_score, recall_score
import sklearn
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import Normalizer
from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing import PowerTransformer
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
warnings.filterwarnings('ignore')

In [2]:
# create any relevant functions
# frame_extractor: extracts the image frame from the list of filenames in the IOP10 file
def frame_extractor(filename):
    parts = filename.split('_')
    frame_number = parts[-1].split('.')[0]
    return frame_number

# time_extractor: extracts the time of day from the list of filenames in the IOP10 file and converts to HH:MM:SS format
def time_extractor(filename):
    parts = filename.split('_')
    time_list = parts[0].split('-')[-3:]
    time_colon_format = ":".join(map(str, time_list))
    return time_colon_format

In [ ]:
#read in the file
hdfFile = h5py.File('pasive_manualclassification_condensed2.h5', 'r')

#create dictionary to store the data
particleTable = {}
print('Reading in hdf5 file')

#list all keys in the file
hdfKeys = list(hdfFile.keys())
N = len(hdfKeys)
# create a list of hdfKeys that will be added to the dataframe later
valid_keys = []
user_list = []

for i, key in enumerate(hdfKeys):
    dset = hdfFile[key]

    #check the 'classifications' attribute
    if 'classifications' in dset.attrs:
        classifications = dset.attrs['classifications']

        for user, value in classifications:
            if value <= 2:
                # add each valid key to the list for the dataframe later
                valid_keys.append(key)
                user_list.append(user)

                for attr in dset.attrs:
                    if attr not in particleTable:
                        particleTable[attr] = []
                    if attr == 'classifications':
                        #we already filtered these, we handle this special
                        particleTable[attr].append( value )
                    else:
                        particleTable[attr].append(dset.attrs[attr])   
    else:
        #there isn't a classification for this particle, wack
        #probably we don't need this else because we're at the end of the loop
        continue     

print(f"Filtered data has been loaded into particleTable with {len(particleTable)} attributes.")

Reading in hdf5 file
Filtered data has been loaded into particleTable with 31 attributes.


In [4]:
# print( particleTable.keys() )
# print( min(particleTable['image']), max(particleTable['image']),)

In [5]:
#for key in particleTable:
#    print( key.ljust(15), len( particleTable[key]) )

print( 'The following classifications were issued')
classificationCounts = {}
for value in particleTable['classifications']:
    if not value in classificationCounts:
        classificationCounts[value] = 0
    classificationCounts[value] += 1

classificationHeader = particleTable['classification_header'][0]
for key in classificationCounts:
    print( '%s %5i'%(classificationHeader[key], classificationCounts[key]) )

# for k in particleTable:
#     print ( k, len(particleTable[k]) )
print( len(hdfKeys))

The following classifications were issued
s 24656
i  5709
b  5427
15008


In [6]:
#This dataframe will contain all data essential to teaching the computer how to identify different classifications of precipitation
#It includes EVERY instance of a classification, so if a particular particle has three classifications, 
#that situation will appear three times, one for each classification

df = pd.DataFrame()
for k in [ 'BrightHist1', 'BrightHist2', 'BrightHist3', 'BrightHist4', 'BrightHist5', 'BrightHist6', 'BrightHist7', 'BrightHist8', 'BrightHist9', 'BrightHist10',
           'Bright_avg', 'Bright_count', 'Bright_max', 'Bright_median', 'classifications', 'e', 'irreg', 'maj', 'maxIrreg', 'min', 'minIrreg', 'prt', 'r']:
    df[k] = particleTable[k] 
#lets also add a column for the valid hdfKeys, but let's clean it up first
framekeys_HDF = [key.split('_')[0] for key in valid_keys]
df['Frame'] = framekeys_HDF
df['User'] = user_list
df.dropna() #this will drop any rows that have a missing value -> prevent any issues with data holes
# display(df)
# print(df.shape)

# Now let's bring in the NSSL csv data for Temperature and pressure HUZZAH
# read the csv into a dataframe
df_csv = pd.read_csv("NSSL1_MW41_output_20230201_185158.csv")
# convert the GPS Time Record row to a datetime object consisting only of the HHMMSS time (cleaner, no reason for a date)
# strip any leading or trailing whitespace
df_csv["GPS Time Record      "] = df_csv["GPS Time Record      "].str.strip() 
# establish current format and convert
df_csv["GPS Time Record      "] = pd.to_datetime(df_csv["GPS Time Record      "], format = '%d-%b-%Y %H:%M:%S')
df_csv['GPS Time Record      '] = df_csv['GPS Time Record      '].dt.strftime('%H:%M:%S')
df_csv.rename(columns={'GPS Time Record      ': 'GPS Time Record'}, inplace=True)
#display(df_csv)

# time to add the associated frame numbers to each time
# i'm thinking I'll add the first frame number for that instance because there are many frame numbers for each second
# first, lets open the txt file and read each line into a list of filenames
with open('IOP10_fileList.txt', 'r') as file:
    filenames = file.readlines()
#clean up the list and get rid of extra spaces
filenames = [filename.strip() for filename in filenames]

# extract the frames from the filenames
frame_numbers = [frame_extractor(filename) for filename in filenames]

# extract the time from the filenames and format it as HH:MM:SS
times = [time_extractor(filename) for filename in filenames]

# create the dataframe
frame_df = pd.DataFrame({'GPS Time Record': times, 'Frame': frame_numbers})

# 18:52:34 is #9489 in iop10, index 9490, need to match
frame_df = frame_df.drop(range(0, 9489))

# get rid of all duplicate timestamps in the filename column (each time will now have exactly one frame mapped to it)
#frame_df = frame_df.drop_duplicates(subset = 'GPS Time Record', keep = 'first')
#frame_df.reset_index(drop = True, inplace = True)
#display(frame_df)

# merge the NSSL file and IOP10 file dataframes based on the aligned timestamps --> add frames to the NSSL df
merged_df = pd.merge(df_csv, frame_df[['GPS Time Record', 'Frame',]], on='GPS Time Record', how='right')
merged_df.drop(0, axis = 0, inplace = True) # drop this first line because there were a handful of slightly anomalous values here
# display(merged_df)
# print(merged_df.shape)

# time for the big reveal.  please let this work
# merge the merged_df with the df of the HDF5 file data, merge using the keys/frame number
final_df = pd.merge(df, merged_df[['Filtered Temperature (K) ', 'Filtered Dewpoint (K) ', 'Filtered Pressure (mb) ', 'Frame', 'GPS Time Record']], on = 'Frame', how = 'inner')
final_df.drop(589, axis = 0, inplace = True) # had some nans

final_df = final_df[final_df['User'] == 3194] #filter by user, manually set

display(final_df)

,BrightHist1,BrightHist2,BrightHist3,BrightHist4,BrightHist5,BrightHist6,BrightHist7,BrightHist8,BrightHist9,BrightHist10,...,min,minIrreg,prt,r,Frame,User,Filtered Temperature (K),Filtered Dewpoint (K),Filtered Pressure (mb),GPS Time Record
2,12.0,127.0,105.0,89.0,70.0,15.0,1.0,0.0,0.0,0.0,...,0.68,0.022,0,0.78,09491,3194,267.9500,267.0068,1076.0000,18:52:34
5,19.0,320.0,359.0,297.0,163.0,94.0,14.0,0.0,0.0,0.0,...,1.18,0.202,3,1.36,09495,3194,267.9500,267.0068,1076.0000,18:52:34
7,9.0,206.0,387.0,306.0,278.0,300.0,255.0,108.0,9.0,0.0,...,1.32,0.248,9,1.66,09495,3194,267.9500,267.0068,1076.0000,18:52:34
9,8.0,190.0,209.0,136.0,78.0,11.0,0.0,0.0,0.0,0.0,...,0.76,0.373,14,0.96,09495,3194,267.9500,267.0068,1076.0000,18:52:34
10,2.0,70.0,199.0,117.0,89.0,37.0,4.0,0.0,0.0,0.0,...,0.86,0.087,39,0.88,09495,3194,267.9500,267.0068,1076.0000,18:52:34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31391,0.0,1.0,24.0,49.0,35.0,30.0,5.0,0.0,0.0,0.0,...,0.41,0.417,3,0.45,15879,3194,252.2203,252.2203,776.6438,18:57:54
31394,1.0,56.0,150.0,68.0,62.0,56.0,45.0,22.0,4.0,0.0,...,0.78,0.310,1,0.86,15881,3194,252.2203,252.2203,776.6438,18:57:54
31395,1.0,84.0,164.0,142.0,67.0,28.0,2.0,0.0,0.0,0.0,...,0.73,0.046,0,0.83,15882,3194,252.2203,252.2203,776.6438,18:57:54
31398,3.0,164.0,193.0,113.0,92.0,17.0,3.0,0.0,0.0,0.0,...,0.84,0.573,3,0.93,15883,3194,252.2203,252.2203,776.6438,18:57:54


In [7]:
#ok, let's machine learn
#first, we split the data into two categories: the inputs and the target variable
X = final_df[['BrightHist1', 'BrightHist2', 'BrightHist3', 'BrightHist4', 'BrightHist5', 'BrightHist6', 'BrightHist7', 'BrightHist8', 'BrightHist9', 'BrightHist10', 
        'Bright_avg', 'Bright_count', 'Bright_max', 'Bright_median', 'e', 'irreg', 'maj', 'maxIrreg', 'min', 'minIrreg', 'r', 'Filtered Temperature (K) ',
        'Filtered Dewpoint (K) ', 'Filtered Pressure (mb) ']]
# TODO keep looking into using temp and dewpoint... the pool of cases is pretty small after all these datasets were merged... more may be needed
# TODO i removed 'prt' because if that's just the image particle number it should be pointless right?
y = final_df['classifications']

#in testing, I found that BrightHist2, 3, and 4 have "*****" string entries at some point, so we have to convert those to nan and drop them
#then, we make sure to rematch the y df to x so that they are the same length and play nice
X = X.apply(pd.to_numeric, errors='coerce')
X = X.dropna()
y = y[X.index]

# print(X.shape)
# print(y.shape)

# Check for non-numeric entries in X --> this is where I found that some columns had strings hidden in there a few times
# def check_non_numeric(df):
#     for column in df.columns:
#         if not pd.to_numeric(df[column], errors='coerce').notna().all():
#             print(f"Non-numeric values found in column: {column}")

# check_non_numeric(X)

#now, we split the data into the training set and the testing set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# normalize the datasets to improve the algorithms
# possible scalers/transformers: StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler, Normalizer, QuantileTransformer, *PowerTransformer*
#PowerTransformer seems to give the highest overall accuracies, but Normalizer gives the absolute highest to RandomForest while hurting the others
scaler = PowerTransformer() 
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
###### Random Forest Model #######
# Create a Random Forest classifier
rf_classifier = RandomForestClassifier(n_estimators=140, random_state=42, bootstrap = True) #does bootstrap use replacement or no?

# Train the model on the training data
rf_classifier.fit(X_train_scaled, y_train)

# Make predictions on the testing data
rf_y_pred = rf_classifier.predict(X_test_scaled)

# Calculate accuracy
accuracy = accuracy_score(y_test, rf_y_pred)
print("Accuracy:", accuracy)

print("Classification Report: \n", classification_report(y_test, rf_y_pred))

print("Confusion Matrix: \n", confusion_matrix(y_test, rf_y_pred))

Accuracy: 0.82109375
Classification Report: 
               precision    recall  f1-score   support

           0       0.85      0.94      0.89      1778
           1       0.79      0.52      0.62       463
           2       0.67      0.61      0.64       319

    accuracy                           0.82      2560
   macro avg       0.77      0.69      0.72      2560
weighted avg       0.82      0.82      0.81      2560

Confusion Matrix: 
 [[1667   50   61]
 [ 187  239   37]
 [ 108   15  196]]


In [9]:
###### Logistic Regression Model ######
#make the model
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

#make some predictions
LR_y_pred = model.predict(X_test_scaled)

#evaluate the model and print out some metrics/the results
accuracy = accuracy_score(y_test, LR_y_pred)
print("Accuracy = ", accuracy)

print("Classification Report: \n", classification_report(y_test, LR_y_pred))

print("Confusion Matrix: \n", confusion_matrix(y_test, LR_y_pred))

Accuracy =  0.811328125
Classification Report: 
               precision    recall  f1-score   support

           0       0.85      0.92      0.88      1778
           1       0.72      0.54      0.62       463
           2       0.69      0.58      0.63       319

    accuracy                           0.81      2560
   macro avg       0.75      0.68      0.71      2560
weighted avg       0.80      0.81      0.80      2560

Confusion Matrix: 
 [[1641   77   60]
 [ 188  250   25]
 [ 113   20  186]]


In [10]:
###### Support Vector Machines ######
#create a SVM classifier
classifier = svm.SVC(kernel='rbf', gamma=0.1, random_state=42) # rbf very slightly beats out poly, but but are essentially tied for the best
#0.7329700272479565
#train the model using the training sets
classifier.fit(X_train_scaled, y_train)
#Predict the response for test dataset
SVM_y_pred = classifier.predict(X_test_scaled)

# Model Accuracy: how often is the classifier correct?
print("Accuracy:",accuracy_score(y_test, SVM_y_pred))
# Model Precision: what percentage of positive tuples are labeled as such?
#print("Precision:",precision_score(y_test, SVM_y_pred, average = 'micro'))
# Model Recall: what percentage of positive tuples are labelled as such?
#print("Recall:",recall_score(y_test, SVM_y_pred, average = 'micro'))
print("Classification Report: \n", classification_report(y_test, SVM_y_pred))

print("Confusion Matrix: \n", confusion_matrix(y_test, SVM_y_pred))

Accuracy: 0.828515625
Classification Report: 
               precision    recall  f1-score   support

           0       0.85      0.95      0.89      1778
           1       0.82      0.54      0.65       463
           2       0.71      0.59      0.64       319

    accuracy                           0.83      2560
   macro avg       0.79      0.69      0.73      2560
weighted avg       0.82      0.83      0.82      2560

Confusion Matrix: 
 [[1685   42   51]
 [ 189  248   26]
 [ 119   12  188]]


In [11]:
###### K Nearest Neighbors Model ######
knn_model = KNeighborsClassifier(n_neighbors = 32, weights = 'distance') # n_neighbors = 32 is the highest I found || random state doesn't work here
knn_model.fit(X_train_scaled, y_train)
#make predictions on the test data
knn_y_pred = knn_model.predict(X_test_scaled)

# Calculate the accuracy of the model
accuracy = accuracy_score(y_test, knn_y_pred)
print("Accuracy:", accuracy)
print("Classification Report: \n", classification_report(y_test, knn_y_pred))

print("Confusion Matrix: \n", confusion_matrix(y_test, knn_y_pred))

Accuracy: 0.80390625
Classification Report: 
               precision    recall  f1-score   support

           0       0.82      0.96      0.88      1778
           1       0.81      0.43      0.56       463
           2       0.69      0.49      0.58       319

    accuracy                           0.80      2560
   macro avg       0.77      0.63      0.67      2560
weighted avg       0.80      0.80      0.78      2560

Confusion Matrix: 
 [[1703   36   39]
 [ 235  198   30]
 [ 151   11  157]]


In [12]:
###### Gaussian Naive Bayes ######
#we have to encode the classes to numeric so that the model runs
#this one works best with the Normalizer scaler

gnb = GaussianNB() #random state doesn't work here

# Train the classifier on the training data
gnb.fit(X_train_scaled, y_train)

# Make predictions on the testing data
gnb_y_pred = gnb.predict(X_test_scaled)

# Calculate the accuracy of the model
accuracy = accuracy_score(y_test, gnb_y_pred)
print("Accuracy: ", accuracy)

print("Classification Report: \n", classification_report(y_test, gnb_y_pred))

print("Confusion Matrix: \n", confusion_matrix(y_test, gnb_y_pred))

Accuracy:  0.634375
Classification Report: 
               precision    recall  f1-score   support

           0       0.87      0.63      0.73      1778
           1       0.53      0.52      0.52       463
           2       0.33      0.84      0.47       319

    accuracy                           0.63      2560
   macro avg       0.57      0.66      0.57      2560
weighted avg       0.74      0.63      0.66      2560

Confusion Matrix: 
 [[1115  207  456]
 [ 123  240  100]
 [  42    8  269]]


In [13]:
# automated random forest tuner - lowkey made it slightly worse
# param_grid = { "n_estimators"      : [250, 300],
#            "criterion"         : ["gini", "entropy"],
#            "max_features"      : [3, 5],
#            "max_depth"         : [10, 20],
#            "min_samples_split" : [2, 4] ,
#            "bootstrap": [True, False]}
# random_search = RandomizedSearchCV(RandomForestClassifier(), param_grid)
# random_search.fit(X_train, y_train)
# print(random_search.best_estimator_)